# Softmax

Softmax 将一组任意实数 logits 转换为概率分布，常作为多分类模型的输出层；每个类别概率均为正，且所有概率之和为 1。

对 logits 向量 $z\in\mathbb{R}^{C}$，第 $i$ 类的概率为：

$$\operatorname{softmax}(z)_i=\frac{e^{z_i}}{\sum_{j=1}^{C}e^{z_j}}.$$

为避免指数计算溢出，可利用 Softmax 的平移不变性，令 $m=\max_j z_j$：

$$\operatorname{softmax}(z)_i=\frac{e^{z_i-m}}{\sum_{j=1}^{C}e^{z_j-m}}.$$

下方实现沿指定维度先减去最大值，再执行指数化与归一化。

In [ ]:
import torch

def softmax_manual(x, dim=-1):
    """
    数值稳定的 Softmax。
    x: 任意形状张量；在 dim 维度上将 logits 转换为概率。常见输入为 [N, C]。
    return: 与 x 形状相同；沿 dim 的元素和为 1。
    """
    # 1. keepdim=True 保留被归约的维度，确保 x_max 可广播回 x。
    #    例如 x 为 [N, C]、dim=-1 时，x_max 为 [N, 1]。
    x_max = x.max(dim=dim, keepdim=True)[0]  # max 返回 (values, indices)，这里只取 values。
    # 平移后的 x - x_max 形状仍为 x.shape；每行最大值变为 0，避免 exp 溢出。
    x_exp = torch.exp(x - x_max)
    
    # 2. 分母形状与 x_max 相同（如 [N, 1]），广播除法后输出形状保持为 x.shape。
    return x_exp / x_exp.sum(dim=dim, keepdim=True)

